# 01 — File Inventory
**Purpose:** Understand what files we have before touching any data.

We have 233 CSV files from an InfluxDB export of the EnFa building BMS.
This notebook answers:
- How many files, how large?
- Any empty or non-CSV files?
- Which signals have the most data (largest files)?

**Rule:** We do NOT open any file contents yet. Filesystem metadata only.

In [ ]:
import sys
from pathlib import Path

# Add the shared library to the path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from zoro_eda.config import load_config
from zoro_eda.paths import resolve_paths

cfg = load_config()
paths = resolve_paths(cfg=cfg)
print(f"Project root : {paths.root}")
print(f"Raw data     : {paths.raw_data}")
print(f"Reports      : {paths.reports}")

## Step 1: Scan filesystem metadata

`scan_files()` walks the data directory and returns a list of dicts — one per file.
It only calls `fpath.stat()` (file size, mtime). No file opens.

In [ ]:
def scan_files(raw_dir: Path) -> list[dict]:
    records = []
    for fpath in raw_dir.iterdir():
        if not fpath.is_file():
            continue
        stat = fpath.stat()
        size_bytes = stat.st_size
        records.append({
            "file_name": fpath.name,
            "extension": fpath.suffix.lower(),
            "size_bytes": size_bytes,
            "size_mb": round(size_bytes / 1_024**2, 3),
            "is_empty": size_bytes == 0,
        })
    return sorted(records, key=lambda r: -r["size_bytes"])

records = scan_files(paths.raw_data)
print(f"Total files : {len(records)}")
print(f"Total size  : {sum(r['size_bytes'] for r in records) / 1e9:.2f} GB")

## Step 2: Extension and empty-file check

In a healthy export, all files should be `.csv` with no empties.
Any other extension would be suspicious.

In [ ]:
from collections import Counter

ext_counts = Counter(r["extension"] for r in records)
empty_files = [r for r in records if r["is_empty"]]

print("Extension breakdown:")
for ext, count in ext_counts.most_common():
    total_mb = sum(r["size_mb"] for r in records if r["extension"] == ext)
    print(f"  {ext:10s}  {count:4d} files   {total_mb:10,.1f} MB")

print(f"\nEmpty files: {len(empty_files)}")
# Expected: 0 empty files, all .csv

## Step 3: Top 20 largest files

The largest files have the most rows — usually high-frequency signals.

**Interpretation:** At ~20-second resolution, a 300 MB file contains roughly:
`300 MB / ~50 bytes per row ≈ 6 million rows ≈ 3.5 years of data`

In [ ]:
print(f"{'File':<55} {'Size (MB)':>10}")
print("-" * 67)
for r in records[:20]:
    print(f"{r['file_name']:<55} {r['size_mb']:>10.1f}")

## Step 4: Save the inventory report

Saved to `reports/data_inventory.csv` so later notebooks and scripts
can load it without re-scanning.

In [ ]:
import csv
paths.ensure_output_dirs()

out_path = paths.reports / "data_inventory.csv"
fields = ["file_name", "extension", "size_bytes", "size_mb", "is_empty"]
with open(out_path, "w", newline="", encoding="utf-8") as fh:
    writer = csv.DictWriter(fh, fieldnames=fields, extrasaction="ignore")
    writer.writeheader()
    writer.writerows(records)

print(f"Saved {len(records)} rows → {out_path}")

## Key findings

| Metric | Value |
|---|---|
| Total files | 233 CSV |
| Total size | ~40.5 GB |
| Empty files | 0 |
| Non-CSV files | 0 |
| Largest file | ~338 MB (real_VerlustleistungEnFa.csv) |

**Why 40 GB matters:** We cannot load this into pandas at once.
Every subsequent notebook reads only the **head** (first 6 KB) or
**tail** (last 4 KB) of each file — never the full content.

**Next:** `02_schema_detection.ipynb` — how are these files formatted?